# Multi-head Attention


In [1]:
import math
from pathlib import Path

import torch
import torch.nn as nn

torch.manual_seed(0)

LECTURE_NOTE_PATH = Path(
    "/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Multi-head Attention.md"
)
PREVIOUS_NOTEBOOK = Path(
    "/Users/montekkundan/Developer/ML/llm/notebooks/scaled_dot_product_attention/lecture_walkthrough.ipynb"
)

def split_heads(x: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq_len, d_model = x.shape
    head_dim = d_model // num_heads
    return x.view(batch, seq_len, num_heads, head_dim).transpose(1, 2)

def merge_heads(x: torch.Tensor) -> torch.Tensor:
    batch, num_heads, seq_len, head_dim = x.shape
    return x.transpose(1, 2).contiguous().view(batch, seq_len, num_heads * head_dim)

def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scores = q @ k.transpose(-1, -2) / math.sqrt(q.size(-1))
    probs = torch.softmax(scores, dim=-1)
    return probs @ v, probs

print(LECTURE_NOTE_PATH)
print('Uses the same scaled attention core as:', PREVIOUS_NOTEBOOK)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Multi-head Attention.md
Uses the same scaled attention core as: /Users/montekkundan/Developer/ML/llm/notebooks/scaled_dot_product_attention/lecture_walkthrough.ipynb


## Demo A: one big projection plus reshape gives many heads


In [2]:
batch, seq_len, d_model, num_heads = 2, 4, 8, 2
x = torch.randn(batch, seq_len, d_model)
q_proj = nn.Linear(d_model, d_model, bias=False)

q = q_proj(x)
q_heads = split_heads(q, num_heads)

print('input shape        :', tuple(x.shape))
print('projected shape    :', tuple(q.shape))
print('after split_heads  :', tuple(q_heads.shape))


input shape        : (2, 4, 8)
projected shape    : (2, 4, 8)
after split_heads  : (2, 2, 4, 4)


## Demo B: multi-head attention end to end


In [3]:
batch, seq_len, d_model, num_heads = 1, 4, 8, 2
x = torch.randn(batch, seq_len, d_model)
q_proj = nn.Linear(d_model, d_model, bias=False)
k_proj = nn.Linear(d_model, d_model, bias=False)
v_proj = nn.Linear(d_model, d_model, bias=False)
o_proj = nn.Linear(d_model, d_model, bias=False)

q = split_heads(q_proj(x), num_heads)
k = split_heads(k_proj(x), num_heads)
v = split_heads(v_proj(x), num_heads)

head_out, head_probs = scaled_dot_product_attention(q, k, v)
merged = merge_heads(head_out)
out = o_proj(merged)

print('per-head probabilities shape:', tuple(head_probs.shape))
print('merged shape                :', tuple(merged.shape))
print('final output shape          :', tuple(out.shape))
print('head 0 attention map:')
print(head_probs[0, 0])
print('head 1 attention map:')
print(head_probs[0, 1])


per-head probabilities shape: (1, 2, 4, 4)
merged shape                : (1, 4, 8)
final output shape          : (1, 4, 8)
head 0 attention map:
tensor([[0.3083, 0.1787, 0.2436, 0.2695],
        [0.2427, 0.2772, 0.2179, 0.2622],
        [0.2298, 0.2444, 0.2832, 0.2426],
        [0.2465, 0.1858, 0.2156, 0.3521]], grad_fn=<SelectBackward0>)
head 1 attention map:
tensor([[0.2416, 0.2221, 0.2634, 0.2729],
        [0.2576, 0.2748, 0.2252, 0.2423],
        [0.2432, 0.2575, 0.2604, 0.2389],
        [0.2627, 0.2604, 0.2329, 0.2440]], grad_fn=<SelectBackward0>)


## Demo C: parameter count stays roughly fixed when d_model stays fixed


In [4]:
def mha_parameter_count(d_model: int, num_heads: int) -> int:
    head_dim = d_model // num_heads
    qkv = 3 * d_model * (num_heads * head_dim)
    out = (num_heads * head_dim) * d_model
    return qkv + out

for heads in [1, 2, 4, 8, 16]:
    print(f'heads={heads:>2} -> parameters={mha_parameter_count(d_model=512, num_heads=heads):,}')


heads= 1 -> parameters=1,048,576
heads= 2 -> parameters=1,048,576
heads= 4 -> parameters=1,048,576
heads= 8 -> parameters=1,048,576
heads=16 -> parameters=1,048,576


## Demo D: many heads means many distributions


In [5]:
x = torch.tensor(
    [
        [[1.0, 0.0, 1.0, 0.0], [0.9, 0.1, 0.2, 0.8], [0.0, 1.0, 0.8, 0.2], [0.1, 0.9, 0.0, 1.0]],
    ]
)
num_heads = 2

Wq = torch.eye(4)
Wk = torch.eye(4)
Wv = torch.eye(4)

q = split_heads(x @ Wq, num_heads)
k = split_heads(x @ Wk, num_heads)
v = split_heads(x @ Wv, num_heads)
_, probs = scaled_dot_product_attention(q, k, v)

print('head 0 attends using the first half of features:')
print(probs[0, 0])
print('head 1 attends using the second half of features:')
print(probs[0, 1])


head 0 attends using the first half of features:
tensor([[0.3385, 0.3154, 0.1669, 0.1791],
        [0.3211, 0.3035, 0.1824, 0.1930],
        [0.1669, 0.1791, 0.3385, 0.3154],
        [0.1824, 0.1930, 0.3211, 0.3035]])
head 1 attends using the second half of features:
tensor([[0.3414, 0.1939, 0.2964, 0.1683],
        [0.1992, 0.2796, 0.2168, 0.3044],
        [0.3044, 0.2168, 0.2796, 0.1992],
        [0.1683, 0.2964, 0.1939, 0.3414]])


## Demo E: MHA vs GQA vs MQA KV-cache size


In [6]:
def kv_cache_elements(batch: int, seq_len: int, num_kv_heads: int, head_dim: int, layers: int) -> int:
    return batch * seq_len * num_kv_heads * head_dim * 2 * layers

batch = 1
seq_len = 8192
num_heads = 32
head_dim = 128
layers = 24

for label, num_kv_heads in [('MHA', 32), ('GQA', 8), ('MQA', 1)]:
    elems = kv_cache_elements(batch, seq_len, num_kv_heads, head_dim, layers)
    print(f'{label}: {elems:,} float values')


MHA: 1,610,612,736 float values
GQA: 402,653,184 float values
MQA: 50,331,648 float values


## Rasbt and nanochat


In [7]:
references = {
    'rasbt': {
        'files': [
            'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb',
            'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/multihead-attention.ipynb',
        ],
        'core': 'Rasbt teaches multi-head attention as repeated scaled attention heads plus concat and output projection, with tensor shapes kept very explicit.',
    },
    'nanochat': {
        'files': [
            'https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py',
        ],
        'core': 'nanochat uses the fused production layout: large QKV projections, reshape into heads, fast attention kernels, and inference-aware KV-cache choices.',
    },
}

for name, info in references.items():
    print(name.upper())
    print('core :', info['core'])
    print('files:')
    for file in info['files']:
        print(' -', file)
    print()


RASBT
core : Rasbt teaches multi-head attention as repeated scaled attention heads plus concat and output projection, with tensor shapes kept very explicit.
files:
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/multihead-attention.ipynb

NANOCHAT
core : nanochat uses the fused production layout: large QKV projections, reshape into heads, fast attention kernels, and inference-aware KV-cache choices.
files:
 - https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py

